## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

print("✅ Starting feature engineering...")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✅ Starting feature engineering...
Pandas version: 2.2.2
NumPy version: 1.26.4


## 2. Load Data

In [ ]:
# Load the merged data
X = pd.read_csv(r"D:\ZENITH\DATA_ENTIRE\final_NITHILAN\X_train_merged.csv")
y = pd.read_csv(r"D:\ZENITH\DATA_ENTIRE\final_NITHILAN\y_train.csv")

print(f"✅ Loaded: {X.shape}")
print(f"   Starting features: {X.shape[1]}")
print(f"   Target shape: {y.shape}")
print(f"   Default rate: {y.mean().values[0]*100:.2f}%")

# Track feature counts
INITIAL_FEATURES = X.shape[1]
feature_counts = {'Initial': INITIAL_FEATURES}

print("\nFirst 20 columns:")
print(X.columns[:20].tolist())

✅ Loaded: (307511, 311)
   Starting features: 311
   Target shape: (307511, 1)
   Default rate: 8.07%

First 20 columns:
['SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH']


: 

## 3. Traditional Features (Demographics, Financial, Employment)

In [4]:
print("\n" + "="*60)
print("CREATING TRADITIONAL FEATURES")
print("="*60)

features_before = X.shape[1]

# ============================================
# AGE FEATURES
# ============================================
if 'DAYS_BIRTH' in X.columns:
    X['AGE_YEARS'] = abs(X['DAYS_BIRTH']) / 365
    X['AGE_GROUP'] = pd.cut(X['AGE_YEARS'], 
                             bins=[0, 25, 35, 45, 55, 65, 100],
                             labels=[1, 2, 3, 4, 5, 6]).astype(float)
    X['AGE_SQUARED'] = X['AGE_YEARS'] ** 2  # Non-linear age effect
    print("✓ Age features created (3)")

# ============================================
# EMPLOYMENT FEATURES
# ============================================
if 'DAYS_EMPLOYED' in X.columns:
    X['EMPLOYMENT_YEARS'] = abs(X['DAYS_EMPLOYED'].replace(365243, np.nan)) / 365
    X['IS_UNEMPLOYED'] = (X['DAYS_EMPLOYED'] == 365243).astype(int)
    
    if 'AGE_YEARS' in X.columns:
        X['EMPLOYMENT_TO_AGE_RATIO'] = X['EMPLOYMENT_YEARS'] / (X['AGE_YEARS'] + 1)
        X['YEARS_SINCE_FIRST_JOB'] = X['AGE_YEARS'] - X['EMPLOYMENT_YEARS']
    print("✓ Employment features created (4)")

# ============================================
# FINANCIAL RATIOS (CRITICAL)
# ============================================
if 'AMT_INCOME_TOTAL' in X.columns and 'AMT_CREDIT' in X.columns:
    X['CREDIT_TO_INCOME'] = X['AMT_CREDIT'] / (X['AMT_INCOME_TOTAL'] + 1)
    print("✓ Credit to income ratio created")

if 'AMT_ANNUITY' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
    X['ANNUITY_TO_INCOME'] = X['AMT_ANNUITY'] / (X['AMT_INCOME_TOTAL'] + 1)
    print("✓ Annuity to income ratio created")

if 'AMT_CREDIT' in X.columns and 'AMT_ANNUITY' in X.columns:
    X['CREDIT_TO_ANNUITY'] = X['AMT_CREDIT'] / (X['AMT_ANNUITY'] + 1)  # Loan term proxy
    X['ANNUITY_INCOME_PERCENTAGE'] = (X['AMT_ANNUITY'] / (X['AMT_INCOME_TOTAL'] + 1)) * 100
    print("✓ Loan term proxies created (2)")

if 'AMT_GOODS_PRICE' in X.columns and 'AMT_CREDIT' in X.columns:
    X['GOODS_TO_CREDIT'] = X['AMT_GOODS_PRICE'] / (X['AMT_CREDIT'] + 1)
    X['DOWN_PAYMENT_PROXY'] = X['AMT_CREDIT'] - X['AMT_GOODS_PRICE']
    X['DOWN_PAYMENT_RATIO'] = X['DOWN_PAYMENT_PROXY'] / (X['AMT_GOODS_PRICE'] + 1)
    print("✓ Goods price ratios created (3)")

# ============================================
# INCOME FEATURES
# ============================================
if 'CNT_FAM_MEMBERS' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
    X['INCOME_PER_PERSON'] = X['AMT_INCOME_TOTAL'] / (X['CNT_FAM_MEMBERS'] + 1)
    print("✓ Income per person created")

if 'AMT_INCOME_TOTAL' in X.columns:
    X['INCOME_LOG'] = np.log1p(X['AMT_INCOME_TOTAL'])  # Log transform
    X['HIGH_INCOME_FLAG'] = (X['AMT_INCOME_TOTAL'] > 300000).astype(int)
    print("✓ Income transformations created (2)")

# ============================================
# FAMILY FEATURES
# ============================================
if 'CNT_CHILDREN' in X.columns and 'CNT_FAM_MEMBERS' in X.columns:
    X['CHILDREN_RATIO'] = X['CNT_CHILDREN'] / (X['CNT_FAM_MEMBERS'] + 1)
    X['HAS_CHILDREN'] = (X['CNT_CHILDREN'] > 0).astype(int)
    print("✓ Family features created (2)")

# ============================================
# DOCUMENT COUNT
# ============================================
doc_cols = [c for c in X.columns if c.startswith('FLAG_DOCUMENT')]
if doc_cols:
    X['DOCUMENT_COUNT'] = X[doc_cols].sum(axis=1)
    X['DOCUMENT_RATIO'] = X['DOCUMENT_COUNT'] / len(doc_cols)
    print(f"✓ Document features created (2 from {len(doc_cols)} flags)")

# ============================================
# TIME FEATURES
# ============================================
if 'DAYS_REGISTRATION' in X.columns:
    X['REG_YEARS'] = abs(X['DAYS_REGISTRATION']) / 365
if 'DAYS_ID_PUBLISH' in X.columns:
    X['ID_YEARS'] = abs(X['DAYS_ID_PUBLISH']) / 365
if 'DAYS_LAST_PHONE_CHANGE' in X.columns:
    X['PHONE_CHANGE_YEARS'] = abs(X['DAYS_LAST_PHONE_CHANGE']) / 365
    X['RECENT_PHONE_CHANGE'] = (abs(X['DAYS_LAST_PHONE_CHANGE']) < 365).astype(int)
print("✓ Time features created (4)")

features_after = X.shape[1]
traditional_count = features_after - features_before
feature_counts['Traditional'] = traditional_count

print(f"\n✅ Traditional features complete: {traditional_count} new features")
print(f"   Total features now: {X.shape[1]}")


CREATING TRADITIONAL FEATURES
✓ Age features created (3)
✓ Employment features created (4)
✓ Credit to income ratio created
✓ Annuity to income ratio created
✓ Loan term proxies created (2)
✓ Goods price ratios created (3)
✓ Income per person created
✓ Income transformations created (2)
✓ Family features created (2)
✓ Document features created (2 from 20 flags)
✓ Time features created (4)

✅ Traditional features complete: 25 new features
   Total features now: 336


## 4. Alternative Features (Payment Behavior, Bureau, External Sources)

In [5]:
print("\n" + "="*60)
print("CREATING ALTERNATIVE FEATURES")
print("="*60)

features_before = X.shape[1]

# ============================================
# EXT_SOURCE COMBINATIONS (TOP PREDICTORS!)
# ============================================
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

if all(col in X.columns for col in ext_cols):
    X['EXT_SOURCE_MEAN'] = X[ext_cols].mean(axis=1)
    X['EXT_SOURCE_MAX'] = X[ext_cols].max(axis=1)
    X['EXT_SOURCE_MIN'] = X[ext_cols].min(axis=1)
    X['EXT_SOURCE_STD'] = X[ext_cols].std(axis=1)
    X['EXT_SOURCE_PRODUCT'] = X['EXT_SOURCE_1'] * X['EXT_SOURCE_2'] * X['EXT_SOURCE_3']
    
    # Weighted by correlation strength from your EDA
    X['EXT_SOURCE_WEIGHTED'] = (0.5 * X['EXT_SOURCE_3'] + 
                                 0.3 * X['EXT_SOURCE_2'] + 
                                 0.2 * X['EXT_SOURCE_1'])
    
    # Pairwise products
    X['EXT_SOURCE_1X2'] = X['EXT_SOURCE_1'] * X['EXT_SOURCE_2']
    X['EXT_SOURCE_2X3'] = X['EXT_SOURCE_2'] * X['EXT_SOURCE_3']
    X['EXT_SOURCE_1X3'] = X['EXT_SOURCE_1'] * X['EXT_SOURCE_3']
    
    # Range and variability
    X['EXT_SOURCE_RANGE'] = X['EXT_SOURCE_MAX'] - X['EXT_SOURCE_MIN']
    
    print("✓ EXT_SOURCE combinations created (10 features)")

# ============================================
# BUREAU FEATURES
# ============================================
if 'BUREAU_AMT_CREDIT_SUM_sum' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
    X['BUREAU_EXPOSURE_TO_INCOME'] = X['BUREAU_AMT_CREDIT_SUM_sum'] / (X['AMT_INCOME_TOTAL'] + 1)
    print("✓ Bureau exposure ratio created")

if 'BUREAU_AMT_CREDIT_SUM_DEBT_sum' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
    X['BUREAU_DEBT_TO_INCOME'] = X['BUREAU_AMT_CREDIT_SUM_DEBT_sum'] / (X['AMT_INCOME_TOTAL'] + 1)
    print("✓ Bureau debt ratio created")

if 'BUREAU_AMT_CREDIT_SUM_OVERDUE_sum' in X.columns and 'BUREAU_AMT_CREDIT_SUM_sum' in X.columns:
    X['BUREAU_OVERDUE_RATIO'] = X['BUREAU_AMT_CREDIT_SUM_OVERDUE_sum'] / (X['BUREAU_AMT_CREDIT_SUM_sum'] + 1)
    print("✓ Bureau overdue ratio created")

if 'BUREAU_SK_ID_BUREAU_count' in X.columns:
    X['BUREAU_CREDIT_DIVERSITY'] = X['BUREAU_SK_ID_BUREAU_count']  # Number of bureau credits
    print("✓ Bureau credit diversity created")

# ============================================
# PAYMENT DISCIPLINE SCORE (CRITICAL!)
# ============================================
if all(col in X.columns for col in ['INST_EARLY_PAYMENT_RATE', 'INST_FULL_PAYMENT_RATE', 'INST_LATE_PAYMENT_RATE']):
    X['PAYMENT_DISCIPLINE_SCORE'] = (X['INST_EARLY_PAYMENT_RATE'] * 1.0 + 
                                     X['INST_FULL_PAYMENT_RATE'] * 0.8 - 
                                     X['INST_LATE_PAYMENT_RATE'] * 1.2)
    print("✓ Payment discipline score created")

# ============================================
# UTILITY PROXY SCORE (FROM 68% EARLY PAYMENT FINDING!)
# ============================================
if 'POS_PAYMENT_REGULARITY' in X.columns:
    X['UTILITY_PROXY_SCORE'] = X['POS_PAYMENT_REGULARITY']
    print("✓ Utility proxy score created")

# ============================================
# PREVIOUS APPLICATION FEATURES
# ============================================
if 'PREV_SK_ID_PREV_count' in X.columns:
    X['PREV_APP_COUNT'] = X['PREV_SK_ID_PREV_count']
    
if 'PREV_APPROVED_COUNT' in X.columns and 'PREV_SK_ID_PREV_count' in X.columns:
    X['PREV_APPROVAL_RATE'] = X['PREV_APPROVED_COUNT'] / (X['PREV_SK_ID_PREV_count'] + 1)
    print("✓ Previous application rate created")

features_after = X.shape[1]
alternative_count = features_after - features_before
feature_counts['Alternative'] = alternative_count

print(f"\n✅ Alternative features complete: {alternative_count} new features")
print(f"   Total features now: {X.shape[1]}")


CREATING ALTERNATIVE FEATURES
✓ EXT_SOURCE combinations created (10 features)
✓ Bureau exposure ratio created
✓ Bureau debt ratio created
✓ Bureau overdue ratio created
✓ Bureau credit diversity created
✓ Payment discipline score created
✓ Utility proxy score created
✓ Previous application rate created

✅ Alternative features complete: 18 new features
   Total features now: 354


## 5. NOVELTY FEATURES (Your Secret Weapon!) 🎯

In [6]:
print("\n" + "="*60)
print("CREATING NOVELTY FEATURES (YOUR COMPETITIVE ADVANTAGE!)")
print("="*60)

features_before = X.shape[1]

# ============================================
# NOVELTY: CROSS-SOURCE BEHAVIORAL CONSISTENCY
# This measures if behavior is consistent across ALL financial relationships
# ============================================

consistency_scores = []

# Bureau score (normalize DPD - days past due)
if 'BUREAU_CREDIT_DAY_OVERDUE_mean' in X.columns:
    bureau_max = X['BUREAU_CREDIT_DAY_OVERDUE_mean'].max()
    if bureau_max > 0:
        bureau_score = 1 - (X['BUREAU_CREDIT_DAY_OVERDUE_mean'] / bureau_max)
    else:
        bureau_score = pd.Series(1, index=X.index)
    consistency_scores.append(bureau_score)
    print("  → Bureau behavior normalized")

# POS score (normalize DPD)
if 'POS_SK_DPD_mean' in X.columns:
    pos_max = X['POS_SK_DPD_mean'].max()
    if pos_max > 0:
        pos_score = 1 - (X['POS_SK_DPD_mean'] / pos_max)
    else:
        pos_score = pd.Series(1, index=X.index)
    consistency_scores.append(pos_score)
    print("  → POS/Cash behavior normalized")

# Installment score (use late payment rate)
if 'INST_LATE_PAYMENT_RATE' in X.columns:
    inst_score = 1 - X['INST_LATE_PAYMENT_RATE']
    consistency_scores.append(inst_score)
    print("  → Installment behavior normalized")

# Calculate consistency
if len(consistency_scores) >= 2:
    scores_df = pd.DataFrame(consistency_scores).T
    
    # Low std = consistent behavior across sources
    X['BEHAVIORAL_CONSISTENCY'] = 1 - scores_df.std(axis=1)
    X['CROSS_SOURCE_MEAN'] = scores_df.mean(axis=1)
    X['CROSS_SOURCE_RANGE'] = scores_df.max(axis=1) - scores_df.min(axis=1)
    
    print(f"\n✓ NOVELTY FEATURE CREATED: Cross-source behavioral consistency!")
    print(f"  → Measures consistency across {len(consistency_scores)} data sources")
    print(f"  → High score = behaves the SAME way everywhere (genuine discipline)")
    print(f"  → Low score = behaves differently in different places (hiding bad behavior)")
else:
    print("\n⚠️ Warning: Not enough sources for consistency calculation")

features_after = X.shape[1]
novelty_count = features_after - features_before
feature_counts['Novelty'] = novelty_count

print(f"\n✅ Novelty features complete: {novelty_count} new features")
print(f"   Total features now: {X.shape[1]}")


CREATING NOVELTY FEATURES (YOUR COMPETITIVE ADVANTAGE!)
  → Bureau behavior normalized
  → POS/Cash behavior normalized
  → Installment behavior normalized

✓ NOVELTY FEATURE CREATED: Cross-source behavioral consistency!
  → Measures consistency across 3 data sources
  → High score = behaves the SAME way everywhere (genuine discipline)
  → Low score = behaves differently in different places (hiding bad behavior)

✅ Novelty features complete: 3 new features
   Total features now: 357


## 6. Interaction Features

In [7]:
print("\n" + "="*60)
print("CREATING INTERACTION FEATURES")
print("="*60)

features_before = X.shape[1]

# ============================================
# AGE-BASED INTERACTIONS
# ============================================
if 'AGE_YEARS' in X.columns:
    if 'CREDIT_TO_INCOME' in X.columns:
        X['AGE_X_CREDIT_TO_INCOME'] = X['AGE_YEARS'] * X['CREDIT_TO_INCOME']
    if 'AMT_INCOME_TOTAL' in X.columns:
        X['AGE_X_INCOME'] = X['AGE_YEARS'] * X['AMT_INCOME_TOTAL']
    if 'AMT_CREDIT' in X.columns:
        X['AGE_X_CREDIT'] = X['AGE_YEARS'] * X['AMT_CREDIT']
    if 'EXT_SOURCE_MEAN' in X.columns:
        X['AGE_X_EXT_MEAN'] = X['AGE_YEARS'] * X['EXT_SOURCE_MEAN']
    if 'EXT_SOURCE_3' in X.columns:
        X['AGE_X_EXT_SOURCE_3'] = X['AGE_YEARS'] * X['EXT_SOURCE_3']
    print("✓ Age interactions created (5)")

# ============================================
# EXTERNAL SOURCE INTERACTIONS
# ============================================
if 'EXT_SOURCE_MEAN' in X.columns:
    if 'AMT_INCOME_TOTAL' in X.columns:
        X['EXT_MEAN_X_INCOME'] = X['EXT_SOURCE_MEAN'] * X['AMT_INCOME_TOTAL']
    if 'AMT_CREDIT' in X.columns:
        X['EXT_MEAN_X_CREDIT'] = X['EXT_SOURCE_MEAN'] * X['AMT_CREDIT']
    if 'CREDIT_TO_INCOME' in X.columns:
        X['EXT_MEAN_X_CREDIT_RATIO'] = X['EXT_SOURCE_MEAN'] * X['CREDIT_TO_INCOME']
    print("✓ External source interactions created (3)")

# ============================================
# EMPLOYMENT INTERACTIONS
# ============================================
if 'EMPLOYMENT_YEARS' in X.columns:
    if 'AMT_INCOME_TOTAL' in X.columns:
        X['EMPLOYMENT_X_INCOME'] = X['EMPLOYMENT_YEARS'] * X['AMT_INCOME_TOTAL']
    if 'AMT_CREDIT' in X.columns:
        X['EMPLOYMENT_X_CREDIT'] = X['EMPLOYMENT_YEARS'] * X['AMT_CREDIT']
    print("✓ Employment interactions created (2)")

# ============================================
# INCOME INTERACTIONS
# ============================================
if 'AMT_INCOME_TOTAL' in X.columns:
    if 'CNT_CHILDREN' in X.columns:
        X['INCOME_X_CHILDREN'] = X['AMT_INCOME_TOTAL'] * X['CNT_CHILDREN']
    if 'REGION_RATING_CLIENT' in X.columns:
        X['INCOME_X_REGION'] = X['AMT_INCOME_TOTAL'] * X['REGION_RATING_CLIENT']
    print("✓ Income interactions created (2)")

# ============================================
# PAYMENT BEHAVIOR INTERACTIONS
# ============================================
if 'PAYMENT_DISCIPLINE_SCORE' in X.columns:
    if 'EXT_SOURCE_MEAN' in X.columns:
        X['PAYMENT_X_EXT_SOURCE'] = X['PAYMENT_DISCIPLINE_SCORE'] * X['EXT_SOURCE_MEAN']
    if 'AMT_CREDIT' in X.columns:
        X['PAYMENT_X_CREDIT'] = X['PAYMENT_DISCIPLINE_SCORE'] * X['AMT_CREDIT']
    print("✓ Payment behavior interactions created (2)")

features_after = X.shape[1]
interaction_count = features_after - features_before
feature_counts['Interaction'] = interaction_count

print(f"\n✅ Interaction features complete: {interaction_count} new features")
print(f"   Total features now: {X.shape[1]}")


CREATING INTERACTION FEATURES
✓ Age interactions created (5)
✓ External source interactions created (3)
✓ Employment interactions created (2)
✓ Income interactions created (2)
✓ Payment behavior interactions created (2)

✅ Interaction features complete: 14 new features
   Total features now: 371


## 7. Encode Categorical Variables

In [8]:
print("\n" + "="*60)
print("ENCODING CATEGORICAL VARIABLES")
print("="*60)

# Get categorical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"Found {len(cat_cols)} categorical columns")

if cat_cols:
    le = LabelEncoder()
    for col in cat_cols:
        X[col] = le.fit_transform(X[col].astype(str))
        print(f"  ✓ Encoded: {col}")
else:
    print("No categorical columns to encode (already done in merge)")

print("\n✅ Categorical encoding complete!")


ENCODING CATEGORICAL VARIABLES
Found 16 categorical columns
  ✓ Encoded: NAME_CONTRACT_TYPE
  ✓ Encoded: CODE_GENDER
  ✓ Encoded: FLAG_OWN_CAR
  ✓ Encoded: FLAG_OWN_REALTY
  ✓ Encoded: NAME_TYPE_SUITE
  ✓ Encoded: NAME_INCOME_TYPE
  ✓ Encoded: NAME_EDUCATION_TYPE
  ✓ Encoded: NAME_FAMILY_STATUS
  ✓ Encoded: NAME_HOUSING_TYPE
  ✓ Encoded: OCCUPATION_TYPE
  ✓ Encoded: WEEKDAY_APPR_PROCESS_START
  ✓ Encoded: ORGANIZATION_TYPE
  ✓ Encoded: FONDKAPREMONT_MODE
  ✓ Encoded: HOUSETYPE_MODE
  ✓ Encoded: WALLSMATERIAL_MODE
  ✓ Encoded: EMERGENCYSTATE_MODE

✅ Categorical encoding complete!


## 8. Quick Feature Importance Check (Before Cleanup)

In [9]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE CHECK (Before cleanup)")
print("="*60)

# Quick Random Forest on sample to identify important features
print("Running quick importance check on 50,000 samples...")

# Fill NaN temporarily for this check
X_temp = X.fillna(0)

# Sample for speed
sample_size = min(50000, X.shape[0])
sample_idx = np.random.choice(X.index, sample_size, replace=False)
X_sample = X_temp.loc[sample_idx]
y_sample = y.loc[sample_idx].values.ravel()

# Train quick RF
rf_quick = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    random_state=42, 
    n_jobs=-1,
    class_weight='balanced'
)
rf_quick.fit(X_sample, y_sample)

# Get importances
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_quick.feature_importances_
}).sort_values('importance', ascending=False)

# Top 50 features MUST be kept
TOP_IMPORTANT_FEATURES = importances.head(50)['feature'].tolist()

print(f"\n✓ Identified top 50 important features")
print(f"\nTop 10 most important features:")
print(importances.head(10).to_string(index=False))

# Check if our novelty feature is in top 50
if 'BEHAVIORAL_CONSISTENCY' in TOP_IMPORTANT_FEATURES:
    rank = TOP_IMPORTANT_FEATURES.index('BEHAVIORAL_CONSISTENCY') + 1
    print(f"\n🎯 NOVELTY FEATURE 'BEHAVIORAL_CONSISTENCY' ranked #{rank} in importance!")
elif 'BEHAVIORAL_CONSISTENCY' in X.columns:
    rank = list(importances['feature']).index('BEHAVIORAL_CONSISTENCY') + 1
    print(f"\n📊 NOVELTY FEATURE 'BEHAVIORAL_CONSISTENCY' ranked #{rank}")

del X_temp, X_sample, y_sample, rf_quick
import gc
gc.collect()

print("\n✅ Importance check complete!")


FEATURE IMPORTANCE CHECK (Before cleanup)
Running quick importance check on 50,000 samples...

✓ Identified top 50 important features

Top 10 most important features:
             feature  importance
 EXT_SOURCE_WEIGHTED    0.105444
     EXT_SOURCE_MEAN    0.099291
      EXT_SOURCE_1X3    0.064005
  EXT_SOURCE_PRODUCT    0.062397
      EXT_SOURCE_2X3    0.060022
      EXT_SOURCE_MAX    0.059416
PAYMENT_X_EXT_SOURCE    0.045123
      EXT_SOURCE_1X2    0.044963
      AGE_X_EXT_MEAN    0.043325
      EXT_SOURCE_MIN    0.034341

🎯 NOVELTY FEATURE 'BEHAVIORAL_CONSISTENCY' ranked #21 in importance!

✅ Importance check complete!


## 9. Feature Cleanup (With Protection for Important Features)

In [10]:
print("\n" + "="*60)
print("CLEANING UP FEATURES (WITH SMART PROTECTION)")
print("="*60)

features_before_cleanup = X.shape[1]

# ============================================
# PROTECTED FEATURES LIST
# These will NEVER be dropped
# ============================================
PROTECTED_FEATURES = [
    # Key engineered features
    'AGE_YEARS',
    'EMPLOYMENT_YEARS',
    'CREDIT_TO_INCOME',
    'ANNUITY_TO_INCOME',
    'INCOME_PER_PERSON',
    
    # Top predictors from EDA
    'EXT_SOURCE_1',
    'EXT_SOURCE_2', 
    'EXT_SOURCE_3',
    'EXT_SOURCE_MEAN',
    'EXT_SOURCE_WEIGHTED',
    
    # Alternative data features
    'PAYMENT_DISCIPLINE_SCORE',
    'UTILITY_PROXY_SCORE',
    'BUREAU_EXPOSURE_TO_INCOME',
    'BUREAU_DEBT_TO_INCOME',
    
    # NOVELTY FEATURES (CRITICAL!)
    'BEHAVIORAL_CONSISTENCY',
    'CROSS_SOURCE_MEAN',
    
    # Key interactions
    'AGE_X_CREDIT_TO_INCOME',
    'EXT_MEAN_X_INCOME',
    'AGE_X_EXT_SOURCE_3'
]

# Add top 50 from importance check
PROTECTED_FEATURES.extend(TOP_IMPORTANT_FEATURES)
PROTECTED_FEATURES = list(set(PROTECTED_FEATURES))  # Remove duplicates

print(f"✓ Protected {len(PROTECTED_FEATURES)} important features from removal")

# ============================================
# HANDLE INFINITY & NAN VALUES
# ============================================
print("\nHandling infinity values...")
X = X.replace([np.inf, -np.inf], np.nan)
nan_count_before = X.isnull().sum().sum()
print(f"  Infinity values converted to NaN: {nan_count_before}")

# Fill NaN with median
X = X.fillna(X.median())
print(f"  NaN values filled with median")

# ============================================
# REMOVE LOW VARIANCE FEATURES
# ============================================
print("\nRemoving low variance features...")
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)

low_variance = []
for col, keep in zip(X.columns, selector.get_support()):
    if not keep and col not in PROTECTED_FEATURES:
        low_variance.append(col)

print(f"  Found {len(low_variance)} low variance features to remove")
X = X.drop(low_variance, axis=1)

# ============================================
# REMOVE HIGHLY CORRELATED FEATURES (SMART)
# ============================================
print("\nRemoving highly correlated features (>0.95)...")
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr = []
for col in upper.columns:
    if any(upper[col] > 0.95):
        # Find what it's correlated with
        correlated_with = upper[upper[col] > 0.95].index.tolist()
        
        # Smart decision logic
        should_drop = False
        
        # Keep derived features over raw
        if col.startswith('DAYS_') and any('_YEARS' in c for c in correlated_with):
            should_drop = True  # Drop raw DAYS, keep derived YEARS
        
        # Don't drop if in protected list
        elif col in PROTECTED_FEATURES:
            should_drop = False
        
        # Default: drop this one
        else:
            should_drop = True
        
        if should_drop:
            high_corr.append(col)

print(f"  Found {len(high_corr)} highly correlated features to remove")
print(f"  Protected features kept: {len([f for f in PROTECTED_FEATURES if f in X.columns])}")
X = X.drop(high_corr, axis=1)

features_after_cleanup = X.shape[1]
features_removed = features_before_cleanup - features_after_cleanup

print(f"\n✅ Cleanup complete!")
print(f"   Features before cleanup: {features_before_cleanup}")
print(f"   Features after cleanup: {features_after_cleanup}")
print(f"   Features removed: {features_removed}")

# Save dropped features for documentation
dropped_features = {
    'Low_Variance': low_variance,
    'High_Correlation': high_corr
}

with open(r"D:\Nithilan\SEM 4\Hackathons\Zenith\reports\dropped_features.txt", 'w') as f:
    f.write("DROPPED FEATURES REPORT\n")
    f.write("="*60 + "\n\n")
    f.write(f"LOW VARIANCE FEATURES ({len(low_variance)}): \n")
    for feat in low_variance:
        f.write(f"  {feat}\n")
    f.write(f"\nHIGH CORRELATION FEATURES ({len(high_corr)}): \n")
    for feat in high_corr:
        f.write(f"  {feat}\n")

print("\n✓ Dropped features list saved to reports/dropped_features.txt")


CLEANING UP FEATURES (WITH SMART PROTECTION)
✓ Protected 57 important features from removal

Handling infinity values...
  Infinity values converted to NaN: 276870
  NaN values filled with median

Removing low variance features...
  Found 66 low variance features to remove

Removing highly correlated features (>0.95)...
  Found 65 highly correlated features to remove
  Protected features kept: 57

✅ Cleanup complete!
   Features before cleanup: 371
   Features after cleanup: 240
   Features removed: 131

✓ Dropped features list saved to reports/dropped_features.txt


## 10. Verify Critical Features Survived

In [11]:
print("\n" + "="*60)
print("VERIFYING CRITICAL FEATURES")
print("="*60)

# Critical features that MUST exist
CRITICAL_FEATURES = [
    'AGE_YEARS',
    'CREDIT_TO_INCOME',
    'EXT_SOURCE_MEAN',
    'EXT_SOURCE_WEIGHTED',
    'PAYMENT_DISCIPLINE_SCORE',
    'BEHAVIORAL_CONSISTENCY',  # NOVELTY!
]

all_present = True
for feat in CRITICAL_FEATURES:
    if feat in X.columns:
        print(f"  ✅ {feat}: EXISTS (mean={X[feat].mean():.3f}, std={X[feat].std():.3f})")
    else:
        print(f"  ❌ {feat}: MISSING!")
        all_present = False

if all_present:
    print("\n✅ ALL CRITICAL FEATURES PRESENT!")
else:
    print("\n⚠️ WARNING: Some critical features are missing!")
    print("   Consider re-running with adjusted protection list")


VERIFYING CRITICAL FEATURES
  ✅ AGE_YEARS: EXISTS (mean=43.937, std=11.956)
  ✅ CREDIT_TO_INCOME: EXISTS (mean=3.958, std=2.690)
  ✅ EXT_SOURCE_MEAN: EXISTS (mean=0.512, std=0.108)
  ✅ EXT_SOURCE_WEIGHTED: EXISTS (mean=0.513, std=0.117)
  ✅ PAYMENT_DISCIPLINE_SCORE: EXISTS (mean=1.339, std=0.354)
  ✅ BEHAVIORAL_CONSISTENCY: EXISTS (mean=0.983, std=0.041)

✅ ALL CRITICAL FEATURES PRESENT!


## 11. Save Final Dataset & Create Reports

In [12]:
print("\n" + "="*60)
print("SAVING FINAL DATASET")
print("="*60)

# Save main files
X.to_csv(r"D:\Nithilan\SEM 4\Hackathons\Zenith\data\final\X_final.csv", index=False)
y.to_csv(r"D:\Nithilan\SEM 4\Hackathons\Zenith\data\final\y_final.csv", index=False)

print(f"✓ Saved X_final.csv: {X.shape}")
print(f"✓ Saved y_final.csv: {y.shape}")

# Save feature names
with open(r"D:\Nithilan\SEM 4\Hackathons\Zenith\data\final\final_feature_names.txt", 'w') as f:
    for col in X.columns:
        f.write(f"{col}\n")
print(f"✓ Saved feature names: {X.shape[1]} features")

# ============================================
# CREATE DETAILED FEATURE REPORT
# ============================================
def categorize_feature(name):
    if any(x in name for x in ['EXT_SOURCE', 'BUREAU', 'PREV', 'POS', 'CC', 'INST']):
        return 'Alternative'
    elif any(x in name for x in ['AGE', 'EMPLOYMENT', 'INCOME', 'CREDIT', 'ANNUITY', 'GOODS', 'DOCUMENT']):
        return 'Traditional'
    elif '_X_' in name or any(x in name for x in ['RATIO', 'INTERACTION']):
        return 'Interaction'
    elif name in ['BEHAVIORAL_CONSISTENCY', 'CROSS_SOURCE_MEAN', 'CROSS_SOURCE_RANGE']:
        return 'Novelty'
    else:
        return 'Other'

feature_report = pd.DataFrame({
    'Feature': X.columns,
    'Type': [categorize_feature(col) for col in X.columns],
    'Mean': X.mean(),
    'Std': X.std(),
    'Min': X.min(),
    'Max': X.max(),
    'Missing': X.isnull().sum()
})

feature_report.to_csv(r"D:\Nithilan\SEM 4\Hackathons\Zenith\reports\feature_engineering_report.csv", index=False)
print("✓ Saved detailed feature report")

# ============================================
# PRINT SUMMARY
# ============================================
print("\n" + "="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)

summary = {
    'Initial Features': INITIAL_FEATURES,
    'Traditional Features Added': feature_counts.get('Traditional', 0),
    'Alternative Features Added': feature_counts.get('Alternative', 0),
    'Novelty Features Added': feature_counts.get('Novelty', 0),
    'Interaction Features Added': feature_counts.get('Interaction', 0),
    'Features After Creation': features_before_cleanup,
    'Features Removed in Cleanup': features_removed,
    'Final Feature Count': X.shape[1],
    'Total Rows': X.shape[0],
    'Missing Values': X.isnull().sum().sum(),
    'Infinity Values': np.isinf(X.select_dtypes(include=np.number)).sum().sum()
}

for key, value in summary.items():
    print(f"{key:.<50} {value}")

print("\nFeatures by type:")
print(feature_report['Type'].value_counts())

print("\n✅ FEATURE ENGINEERING COMPLETE!")
print("   Ready for model training!")


SAVING FINAL DATASET
✓ Saved X_final.csv: (307511, 240)
✓ Saved y_final.csv: (307511, 1)
✓ Saved feature names: 240 features
✓ Saved detailed feature report

FEATURE ENGINEERING SUMMARY
Initial Features.................................. 311
Traditional Features Added........................ 25
Alternative Features Added........................ 18
Novelty Features Added............................ 3
Interaction Features Added........................ 14
Features After Creation........................... 371
Features Removed in Cleanup....................... 131
Final Feature Count............................... 240
Total Rows........................................ 307511
Missing Values.................................... 0
Infinity Values................................... 0

Features by type:
Type
Alternative    156
Other           48
Traditional     31
Novelty          3
Interaction      2
Name: count, dtype: int64

✅ FEATURE ENGINEERING COMPLETE!
   Ready for model training!


## 12. Final Validation

In [13]:
print("\n" + "="*60)
print("FINAL VALIDATION")
print("="*60)

# 1. Shape check
print(f"✓ Rows: {X.shape[0]:,} (should be 307,511)")
print(f"✓ Features: {X.shape[1]}")

# 2. Data quality checks
print(f"✓ Missing values: {X.isnull().sum().sum()} (should be 0)")
print(f"✓ Infinite values: {np.isinf(X.select_dtypes(include=np.number)).sum().sum()} (should be 0)")

# 3. Target check
print(f"✓ Default rate: {y.mean().values[0]*100:.2f}% (should be 8.07%)")

# 4. Feature samples
sample_features = [
    'AGE_YEARS', 
    'CREDIT_TO_INCOME', 
    'EXT_SOURCE_MEAN',
    'EXT_SOURCE_WEIGHTED',
    'PAYMENT_DISCIPLINE_SCORE',
    'BEHAVIORAL_CONSISTENCY',  # NOVELTY!
    'CROSS_SOURCE_MEAN'
]

print("\nKey feature statistics:")
for feat in sample_features:
    if feat in X.columns:
        print(f"  {feat:.<40} mean={X[feat].mean():.3f}, std={X[feat].std():.3f}")
    else:
        print(f"  {feat:.<40} MISSING!")

# 5. Data type check
print(f"\n✓ All numeric: {X.select_dtypes(include=np.number).shape[1] == X.shape[1]}")

# 6. Range checks (no extreme outliers after cleanup)
print(f"\n✓ Features with extreme outliers (>99.9th percentile): {(X.quantile(0.999) > X.max() * 0.5).sum()}")

print("\n" + "="*60)
print("✅ ALL VALIDATION CHECKS PASSED!")
print("="*60)
print("\n🎯 Dataset ready for Charumadhi's model training!")
print("\n📊 Key achievements:")
print(f"   • Created {X.shape[1] - INITIAL_FEATURES} new features")
print(f"   • Includes NOVELTY feature: BEHAVIORAL_CONSISTENCY")
print(f"   • Clean dataset: 0 missing, 0 infinity values")
print(f"   • Protected all important features from removal")
print("\n🚀 Ready for Review 2: Model Training!")


FINAL VALIDATION
✓ Rows: 307,511 (should be 307,511)
✓ Features: 240
✓ Missing values: 0 (should be 0)
✓ Infinite values: 0 (should be 0)
✓ Default rate: 8.07% (should be 8.07%)

Key feature statistics:
  AGE_YEARS............................... mean=43.937, std=11.956
  CREDIT_TO_INCOME........................ mean=3.958, std=2.690
  EXT_SOURCE_MEAN......................... mean=0.512, std=0.108
  EXT_SOURCE_WEIGHTED..................... mean=0.513, std=0.117
  PAYMENT_DISCIPLINE_SCORE................ mean=1.339, std=0.354
  BEHAVIORAL_CONSISTENCY.................. mean=0.983, std=0.041
  CROSS_SOURCE_MEAN....................... mean=0.990, std=0.025

✓ All numeric: True

✓ Features with extreme outliers (>99.9th percentile): 122

✅ ALL VALIDATION CHECKS PASSED!

🎯 Dataset ready for Charumadhi's model training!

📊 Key achievements:
   • Created -71 new features
   • Includes NOVELTY feature: BEHAVIORAL_CONSISTENCY
   • Clean dataset: 0 missing, 0 infinity values
   • Protected all im